# ATP MATCHES

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("silver_atp_matches").getOrCreate()
except Exception as e:
    print(e)

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [4]:
tb_atp_matches = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "bronze.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

## Players

In [5]:
winners = (
        tb_atp_matches 
        .select(
            f.col("winner_id").alias("PLAYER_ID"),
            f.when(
                f.col("winner_name").contains("Shevchenko"), "Aleksandr Shevchenko"
            ).otherwise(f.col("winner_name")).alias("PLAYER_NAME"),
            f.col("winner_hand").alias("PLAYER_HAND"),
            f.col("winner_ht").cast('int').alias("PLAYER_HEIGHT"),
            f.col("winner_ioc").alias("PLAYER_COUNTRY"),
            f.to_date(f.col("tourney_date").cast("string"), "yyyyMMdd").alias("MATCH_DATE"),
            f.col("winner_age").cast("double").alias("PLAYER_AGE")
        )
        .distinct()
    )

In [6]:
losers = (
    tb_atp_matches 
    .select(
        f.col("loser_id").alias("PLAYER_ID"),
        f.when(
            f.col("loser_name").contains("Shevchenko"), "Aleksandr Shevchenko"
        ).otherwise(f.col("loser_name")).alias("PLAYER_NAME"),
        f.col("loser_hand").alias("PLAYER_HAND"),
        f.col("loser_ht").cast("int").alias("PLAYER_HEIGHT"),
        f.col("loser_ioc").alias("PLAYER_COUNTRY"),
        f.to_date(f.col("tourney_date").cast("string"), "yyyyMMdd").alias("MATCH_DATE"),
        f.col("loser_age").cast("double").alias("PLAYER_AGE")
    )
)

In [25]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("PLAYER_ID").orderBy(f.col("MATCH_DATE").desc())

df = (
    winners.unionByName(losers)
    .withColumn(
        "PLAYER_BIRTH_DATE",
        f.expr("date_sub(MATCH_DATE, cast(PLAYER_AGE * 365.25 as int))")
    )
    .withColumn("row_num", f.row_number().over(window_spec))
    .filter(f.col("row_num") == 1)
    .select(
        "PLAYER_ID",
        "PLAYER_NAME",
        "PLAYER_HAND",
        "PLAYER_HEIGHT",
        "PLAYER_COUNTRY",
        "PLAYER_BIRTH_DATE"
    )
)

In [26]:
df.where("PLAYER_NAME = 'Joao Fonseca'")

PLAYER_ID,PLAYER_NAME,PLAYER_HAND,PLAYER_HEIGHT,PLAYER_COUNTRY,PLAYER_BIRTH_DATE
211663,Joao Fonseca,R,185,BRA,2006-08-31
F0FV,Joao Fonseca,R,188,BRA,2006-08-21


## Save dataframe

### Local

In [83]:
df.toPandas().to_csv(
    r"../../data/silver/tb_atp_players.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [84]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("append")
    .save()
)